## PythonによるExcel操作自動化の基本

In [2]:
import sys
!{sys.executable} -m pip install pandas

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ----------- ---------------------------- 2.9/9.7 MB 21.0 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.7 MB 28.0 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 24.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   -------------------- ------------------- 6.3/12.3 MB 35.1 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 33.5 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [5]:
import sys
!{sys.executable} -m pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from openpyxl import Workbook, load_workbook

# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')
# データフレームを表示して確認
df.head()


,カテゴリー,商品コード,商品名,売上日,単価,数量,原価
0,食品,1001,りんご,2023-01-01,200,50,120
1,食品,1002,バナナ,2023-01-01,150,100,80
2,食品,1003,牛乳,2023-01-02,180,80,100
3,衣服,2001,Tシャツ,2023-01-02,1500,20,800
4,衣服,2002,ジーンズ,2023-01-03,5000,10,2500


In [7]:
# 2. データをLLM用にテキスト形式に変換
# データフレーム全体を文字列に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"
# 表示して確認
print(prompt_text)


売上データ:
    カテゴリー 商品コード      商品名         売上日    単価   数量    原価
0      食品  1001      りんご  2023-01-01   200   50   120
1      食品  1002      バナナ  2023-01-01   150  100    80
2      食品  1003       牛乳  2023-01-02   180   80   100
3      衣服  2001     Tシャツ  2023-01-02  1500   20   800
4      衣服  2002     ジーンズ  2023-01-03  5000   10  2500
..    ...   ...      ...         ...   ...  ...   ...
235    衣服  2077   レインパンツ  2023-04-28  2000   18  1000
236    食品  1085      ザクロ  2023-04-29   600   40   300
237   日用品  3077    バスブラシ  2023-04-29   400   60   200
238    衣服  2078  レインシューズ  2023-04-30  2500   15  1250
239    食品  1086    ココナッツ  2023-04-30   300   80   150

[240 rows x 7 columns]
この売上データの傾向を分析してください。


In [8]:
# 3. OpenAI APIの呼び出し

# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())


売上データの分析を行うにあたり、以下のポイントに焦点を当てていただくと、傾向やインサイトを明確に引き出すことができるでしょう。

### 1. 売上の全体トレンド
- **売上日別の売上高**を集計して、時間の経過に伴う売上のトレンドを可視化します。特定の期間（例えば、月ごとや週ごと）で売上が増加または減少している場合、その要因を探ります。

### 2. カテゴリー別の売上分析
- 各カテゴリー（食品、衣服、日用品）ごとに売上高を集計し、どのカテゴリーが最も売上を伸ばしているかを分析します。これにより、人気のカテゴリーや商品群を特定できます。

### 3. 商品別のパフォーマンス
- 各商品の売上を見て、トップセラーとワーストセラーの商品を特定します。特に、利益率（単価 - 原価）も考慮に入れ、売上だけでなく利益に基づいたパフォーマンスを評価します。

### 4. 季節性の分析
- 季節ごとの売上パターンを分析し、特定の季節やイベント（例えば、年末商戦や夏休み）での売上の変動を把握します。

### 5. 価格と数量の関係
- 単価と数量の関係を調べ、価格変更が売上にどのように影響しているかを分析します。

### 6. 原価と利益の分析
- 各商品の原価に対する利益率を計算し、原価が高いが売上が高い商品と、その逆のパターンを特定します。これにより、コスト管理やプロモーション戦略の見直しに繋がるかもしれません。

### 7. 顧客セグメントの理解
- 将来的に顧客データが入手できれば、どの顧客セグメントがどのカテゴリーや商品を好むのかを分析し、ターゲットマーケティングを行う基盤を築きます。

### 結果の可視化
これらのインサイトをもとに、グラフやチャートを用いて結果を視覚的に表現することをおすすめします。例えば、売上の推移を折れ線グラフで示したり、カテゴリー別の売上を円グラフで表示することが考えられます。

これらの分析を通じて、企業の戦略的な意思決定に役立つ具体的なインサイトを得ることができます。


In [9]:
# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])
print(df_out)


                                                   結果
0   売上データの分析を行うにあたり、以下のポイントに焦点を当てていただくと、傾向やインサイトを明...
1                                                    
2                                    ### 1. 売上の全体トレンド
3   - **売上日別の売上高**を集計して、時間の経過に伴う売上のトレンドを可視化します。特定の...
4                                                    
5                                  ### 2. カテゴリー別の売上分析
6   - 各カテゴリー（食品、衣服、日用品）ごとに売上高を集計し、どのカテゴリーが最も売上を伸ばし...
7                                                    
8                                  ### 3. 商品別のパフォーマンス
9   - 各商品の売上を見て、トップセラーとワーストセラーの商品を特定します。特に、利益率（単価 ...
10                                                   
11                                      ### 4. 季節性の分析
12  - 季節ごとの売上パターンを分析し、特定の季節やイベント（例えば、年末商戦や夏休み）での売上...
13                                                   
14                                    ### 5. 価格と数量の関係
15          - 単価と数量の関係を調べ、価格変更が売上にどのように影響しているかを分析します。
16                                                   
17                          

In [10]:
# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)


In [11]:
# ワークフロー化
print("処理を開始します。")

# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')

# 2. データをLLM用にテキスト形式に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"

# 3. OpenAI APIの呼び出し
# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"
# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])

# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

print("Excelファイルに分析結果を保存しました。")


処理を開始します。
Excelファイルに分析結果を保存しました。
